# E14 假设检验：t / χ² / 单因素 ANOVA

> 覆盖矩阵：统计推断 · `docs/ANALYSIS_COVERAGE.md`

## 你将学到
1. 正态 / 方差齐性检查再选检验。
2. 独立样本 t、Welch、Mann–Whitney 对照。
3. 列联表 χ² 与期望频数警告。
4. 单因素 ANOVA + 事后两两比较意识（多重比较）。
5. **wanglaoshi 缺口**：无 `StatsToolkit` 统一封装与中文适用性报告。


In [ ]:
QUICK = True
RANDOM_STATE = 42
from wanglaoshi.runtime import ready
ready(seed=RANDOM_STATE)


## 1. 合成多组数据 + EDA

In [ ]:
import numpy as np
import pandas as pd
from wanglaoshi.analyze import DataAnalyzer
from scipy import stats

rng = np.random.default_rng(RANDOM_STATE)
# 三渠道转化金额（略偏）
channels = []
for name, mu in [('organic', 40), ('paid', 48), ('email', 42)]:
    x = rng.normal(mu, 12, 120)
    channels.append(pd.DataFrame({'channel': name, 'revenue': x, 'device': rng.choice(['ios', 'android'], 120)}))
df = pd.concat(channels, ignore_index=True)
# 类别：是否高价值
df['high_value'] = (df['revenue'] > 50).astype(int)
print(df.groupby('channel')['revenue'].describe()[['mean', 'std', 'count']])
print(DataAnalyzer(df).get_basic_info().get('shape'))


## 2. 两独立样本 t（Welch）与效应量

In [ ]:
a = df.loc[df.channel == 'organic', 'revenue']
b = df.loc[df.channel == 'paid', 'revenue']
# Shapiro（大样本仅作提示）
print('shapiro organic p≈', stats.shapiro(a.sample(min(50, len(a)), random_state=RANDOM_STATE)).pvalue)
tt = stats.ttest_ind(a, b, equal_var=False)
# Cohen's d（pooled）
pooled = np.sqrt(((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / (len(a) + len(b) - 2))
d = (b.mean() - a.mean()) / pooled
print('Welch t=', tt.statistic, 'p=', tt.pvalue, "Cohen's d=", d)
mw = stats.mannwhitneyu(a, b, alternative='two-sided')
print('Mann-Whitney U p=', mw.pvalue)


## 3. χ² 独立性（渠道 × 高价值）

In [ ]:
tab = pd.crosstab(df['channel'], df['high_value'])
print(tab)
chi2, p, dof, exp = stats.chi2_contingency(tab)
print('chi2', chi2, 'p', p, 'min expected', exp.min())
if exp.min() < 5:
    print('警告：期望频数过低，考虑合并类别或 Fisher')


## 4. 单因素 ANOVA

In [ ]:
groups = [g['revenue'].to_numpy() for _, g in df.groupby('channel')]
f_stat, p_anova = stats.f_oneway(*groups)
print('ANOVA F=', f_stat, 'p=', p_anova)
# 简易事后：三对 Welch（未校正——课内强调需校正）
from itertools import combinations
for x, y in combinations(df['channel'].unique(), 2):
    gx = df.loc[df.channel == x, 'revenue']
    gy = df.loc[df.channel == y, 'revenue']
    p = stats.ttest_ind(gx, gy, equal_var=False).pvalue
    print(f'{x} vs {y}: raw p={p:.4f}（展示用，生产请 FDR/Tukey）')


## 5. 交付

In [ ]:
from pathlib import Path
out = Path('outputs/14_hypothesis_testing')
out.mkdir(parents=True, exist_ok=True)
pd.DataFrame([{
    'welch_p': tt.pvalue, 'cohens_d': d, 'anova_p': p_anova, 'chi2_p': p
}]).to_csv(out / 'stats_summary.csv', index=False)
print('saved', out)


## 附录
- 作业：对 `device` 做分层检验；讨论辛普森悖论。
- 包优化：`StatsToolkit.recommend_test(df, y, group)` → 适用性检查 + 效应量 + 中文解读。
